# Naive Bayes — Bag-of-Words vs TF-IDF

Step 2 of the comparison. `MultinomialNB` models each feature as a count drawn from a multinomial
distribution — its assumptions match raw term-frequency counts (Bag-of-Words), not TF-IDF's
re-weighted real values. That's the textbook claim; this notebook checks it against real
cross-validated numbers on this dataset rather than just repeating it.

Both vectorizers use `ngram_range=(1, 2)`, not unigrams only. Stopwords (including negation words
like `không`, `chưa`) were deliberately kept in `260106_MachineLearningForNlp/notebooks/01_...` —
on unigrams alone that only half helps, since "không thích" (don't like) would still contribute
`không` and `thích` as two independent features, and `thích` alone still reads positive. Bigrams
let `không_thích` exist as its own feature. See `Personal Note.md` ("Vectorization and evaluation
methodology") for why full negation-scope tagging isn't used instead — bigrams get most of the
practical benefit without needing a scope-detection rule this project doesn't have punctuation left
to anchor on anyway (Step 1 strips it before this would run).

No separate train/test split — `StratifiedKFold` cross-validation instead, so the comparison isn't
sensitive to which rows happened to land in one particular split.

## Imports

In [1]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import MultinomialNB

## Load Preprocessed Data

In [2]:
PROCESSED_DIR = os.path.join("..", "data", "processed")
OUTPUTS_DIR = os.path.join("..", "data", "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(PROCESSED_DIR, "reviews.parquet"))
X = df["clean_comment"].values
y = df["label"].values

print(df.shape)
df["label"].value_counts(normalize=True).sort_index()

## Cross-Validation Setup

Same seed as every other notebook that uses `StratifiedKFold` in this project (`02`, `03`) — same
data, same seed, same folds, so all methods are compared on identical splits without persisting
fold indices to disk.

In [3]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

## One CV Loop, Reused For Both Vectorizers

`min_df=2` drops terms that appear in only a single training document — with `ngram_range=(1, 2)`
on a corpus this size (≈2,432 training rows per fold), most singleton bigrams are one-off noise
rather than a real repeated pattern (same reasoning as `min_df` in
`260106_Scikit-learnTextFeatureExtraction`, just a lower cutoff here since this corpus is far
smaller than that project's 184K-article one).

In [4]:
def run_cv(vectorizer_factory, model_factory, X, y, skf):
    fold_metrics = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        vectorizer = vectorizer_factory()
        X_train = vectorizer.fit_transform(X[train_idx])
        X_val = vectorizer.transform(X[val_idx])
        y_train, y_val = y[train_idx], y[val_idx]

        model = model_factory()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_val, y_pred, average="macro", zero_division=0
        )
        fold_metrics.append({
            "fold": fold,
            "vocab_size": len(vectorizer.vocabulary_),
            "accuracy": accuracy_score(y_val, y_pred),
            "precision_macro": precision,
            "recall_macro": recall,
            "f1_macro": f1,
        })
    return fold_metrics


def summarize(fold_metrics: list[dict], method: str, vectorizer_desc: str) -> dict:
    keys = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
    mean = {k: float(np.mean([f[k] for f in fold_metrics])) for k in keys}
    std = {k: float(np.std([f[k] for f in fold_metrics])) for k in keys}
    return {
        "method": method,
        "vectorizer": vectorizer_desc,
        "model": "MultinomialNB",
        "n_splits": N_SPLITS,
        "folds": fold_metrics,
        "mean": mean,
        "std": std,
    }

## Naive Bayes + Bag-of-Words

In [5]:
bow_folds = run_cv(
    vectorizer_factory=lambda: CountVectorizer(ngram_range=(1, 2), min_df=2),
    model_factory=MultinomialNB,
    X=X, y=y, skf=skf,
)
nb_bow_results = summarize(bow_folds, method="naive_bayes_bow", vectorizer_desc="CountVectorizer(ngram_range=(1,2), min_df=2)")

print(f"vocab size per fold: {[f['vocab_size'] for f in bow_folds]}")
for k, v in nb_bow_results["mean"].items():
    print(f"{k}: {v:.4f} ± {nb_bow_results['std'][k]:.4f}")

## Naive Bayes + TF-IDF

In [6]:
tfidf_folds = run_cv(
    vectorizer_factory=lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    model_factory=MultinomialNB,
    X=X, y=y, skf=skf,
)
nb_tfidf_results = summarize(tfidf_folds, method="naive_bayes_tfidf", vectorizer_desc="TfidfVectorizer(ngram_range=(1,2), min_df=2)")

print(f"vocab size per fold: {[f['vocab_size'] for f in tfidf_folds]}")
for k, v in nb_tfidf_results["mean"].items():
    print(f"{k}: {v:.4f} ± {nb_tfidf_results['std'][k]:.4f}")

## Bag-of-Words vs TF-IDF, Side By Side

The actual check of the textbook claim this notebook opened with — not asserted, read off real
cross-validated numbers.

In [7]:
comparison = pd.DataFrame({
    "bow_mean": nb_bow_results["mean"],
    "bow_std": nb_bow_results["std"],
    "tfidf_mean": nb_tfidf_results["mean"],
    "tfidf_std": nb_tfidf_results["std"],
})
comparison

## Saving Metrics

In [8]:
bow_path = os.path.join(OUTPUTS_DIR, "nb_bow_metrics.json")
tfidf_path = os.path.join(OUTPUTS_DIR, "nb_tfidf_metrics.json")

with open(bow_path, "w", encoding="utf-8") as f:
    json.dump(nb_bow_results, f, ensure_ascii=False, indent=2)
with open(tfidf_path, "w", encoding="utf-8") as f:
    json.dump(nb_tfidf_results, f, ensure_ascii=False, indent=2)

print(f"Saved -> {bow_path}")
print(f"Saved -> {tfidf_path}")